# 01 — Avaliações de e-commerce: exploração do texto

**Projeto:** prever se um cliente recomendaria o produto a partir do que ele escreveu
**Fonte:** [B2W-Reviews01](https://github.com/americanas-tech/b2w-reviews01) — mais de 130 mil avaliações
da Americanas.com entre janeiro e maio de 2018 (licença CC BY-NC-SA 4.0)

**Objetivo deste notebook:** entender os dados e o texto antes de modelar — distribuição das notas,
relação entre nota e recomendação, vocabulário de cada classe e qualidade dos registros.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)
sns.set_theme(style="whitegrid", context="notebook")

PASTA_RAW = Path("../data/raw")
PASTA_PROCESSED = Path("../data/processed")
PASTA_FIGURAS = Path("../reports/figures")
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

def salvar(fig, nome):
    fig.savefig(PASTA_FIGURAS / f"{nome}.png", dpi=150, bbox_inches="tight")

CORES = {"Não recomenda": "#e76f51", "Recomenda": "#2a9d8f"}

# O arquivo é separado por vírgula, em UTF-8, com aspas duplas nos textos
df = pd.read_csv(PASTA_RAW / "B2W-Reviews01.csv", parse_dates=["submission_date"])
print(f"{len(df):,} avaliações | {df['submission_date'].min():%d/%m/%Y} a {df['submission_date'].max():%d/%m/%Y}")
df.head(3)

## 1. Estrutura e completude

In [ ]:
resumo = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "preenchidos": df.notna().sum(),
    "% nulos": (df.isna().mean() * 100).round(1),
    "valores distintos": df.nunique(),
})
resumo

A marca do produto costuma faltar em boa parte dos registros, e algumas avaliações não têm texto.
Como o texto é justamente a entrada do modelo, essas linhas serão descartadas.

## 2. As duas variáveis-alvo

A base tem dois rótulos: a nota de 1 a 5 estrelas e a resposta "recomendaria a um amigo".
Vale ver se são redundantes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

notas = df["overall_rating"].value_counts().sort_index()
axes[0].bar(notas.index, notas.values, color="#264653")
axes[0].set_title("Distribuição das notas")
axes[0].set_xlabel("Estrelas")
for x, v in notas.items():
    axes[0].text(x, v, f"{v / len(df):.0%}", ha="center", va="bottom", fontsize=9)

recomenda = df["recommend_to_a_friend"].value_counts()
axes[1].bar(["Não", "Sim"], [recomenda.get("No", 0), recomenda.get("Yes", 0)],
            color=[CORES["Não recomenda"], CORES["Recomenda"]])
axes[1].set_title("Recomendaria a um amigo?")
for i, v in enumerate([recomenda.get("No", 0), recomenda.get("Yes", 0)]):
    axes[1].text(i, v, f"{v / len(df):.0%}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
salvar(fig, "01_distribuicao_alvos")
plt.show()

In [ ]:
cruzamento = pd.crosstab(df["overall_rating"], df["recommend_to_a_friend"], normalize="index")
(cruzamento * 100).round(1).rename(columns={"No": "% não recomenda", "Yes": "% recomenda"})

Os rótulos andam juntos, como esperado, mas não são a mesma coisa: nas notas intermediárias
há gente que recomenda e gente que não. São justamente esses casos que tornam o problema interessante —
e são os que um modelo simples tende a errar.

**Escolha do alvo:** vamos prever `recommend_to_a_friend`. É binário, tem interpretação de negócio direta
(o cliente virou promotor ou não) e evita o problema de tratar a nota como se as distâncias entre
estrelas fossem iguais.

## 3. Como é o texto

In [ ]:
df = df.dropna(subset=["review_text", "recommend_to_a_friend"]).copy()
df["recomenda"] = (df["recommend_to_a_friend"] == "Yes").astype(int)
df["texto"] = df["review_title"].fillna("") + " " + df["review_text"]
df["n_caracteres"] = df["review_text"].str.len()
df["n_palavras"] = df["review_text"].str.split().str.len()

print(f"Avaliações com texto: {len(df):,}")
df[["n_caracteres", "n_palavras"]].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(df["n_palavras"].clip(upper=120), bins=40, color="#264653", ax=axes[0])
axes[0].set_title("Tamanho das avaliações")
axes[0].set_xlabel("Palavras (acima de 120 agrupadas)")

dados = df.assign(rotulo=np.where(df["recomenda"] == 1, "Recomenda", "Não recomenda"))
sns.boxplot(data=dados, x="rotulo", y="n_palavras", hue="rotulo", palette=CORES,
            legend=False, showfliers=False, ax=axes[1])
axes[1].set_title("Tamanho por classe")
axes[1].set_xlabel("")
axes[1].set_ylabel("Palavras")

plt.tight_layout()
salvar(fig, "02_tamanho_texto")
plt.show()

df.groupby("recomenda")["n_palavras"].median().rename({0: "Não recomenda", 1: "Recomenda"})

Quem não recomenda costuma escrever mais: reclamação exige explicação, elogio não.
Isso já é um sinal aproveitável pelo modelo, mesmo antes de olhar as palavras.

## 4. Vocabulário de cada classe

Comparamos a frequência relativa de cada termo entre as duas classes. Usamos uma lista de
stopwords em português (palavras muito comuns, como artigos e preposições) para não poluir o resultado.

In [ ]:
STOPWORDS_PT = """a as o os um uma uns umas de do da dos das em no na nos nas por para com sem sob sobre
ao aos à às e ou mas que se como quando onde qual quais quanto porque pois então já ainda também
eu tu ele ela nós vós eles elas me te lhe nos vos lhes meu minha meus minhas seu sua seus suas
este esta isto esse essa isso aquele aquela aquilo ser estar ter haver foi era são é está estão tem têm
tinha foram sendo muito mais menos bem mal todo toda todos todas nada tudo algum alguma nenhum
até depois antes agora aqui ali lá só apenas cada mesmo mesma outro outra pra pro numa num
lhe nem talvez sim não da's""".split()

def preparar_texto(serie: pd.Series) -> pd.Series:
    return (serie.str.lower()
            .str.replace(r"http\S+", " ", regex=True)
            .str.replace(r"[^a-zà-ú\s]", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip())

df["texto_limpo"] = preparar_texto(df["texto"])

vetorizador = CountVectorizer(stop_words=STOPWORDS_PT, min_df=20, max_features=5000)
matriz = vetorizador.fit_transform(df["texto_limpo"])
termos = np.array(vetorizador.get_feature_names_out())

mascara = df["recomenda"].values == 1
freq_pos = np.asarray(matriz[mascara].sum(axis=0)).ravel() / mascara.sum()
freq_neg = np.asarray(matriz[~mascara].sum(axis=0)).ravel() / (~mascara).sum()

# Razão suavizada: quantas vezes o termo é mais comum em uma classe que na outra
comparacao = pd.DataFrame({
    "termo": termos,
    "freq_recomenda": freq_pos,
    "freq_nao_recomenda": freq_neg,
    "razao": (freq_neg + 0.001) / (freq_pos + 0.001),
})
comparacao["ocorrencias"] = np.asarray(matriz.sum(axis=0)).ravel()
comparacao = comparacao[comparacao["ocorrencias"] >= 100]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (dados, titulo, cor, crescente) in zip(axes, [
    (comparacao.nlargest(15, "razao"), "Mais típicos de quem NÃO recomenda", CORES["Não recomenda"], False),
    (comparacao.nsmallest(15, "razao"), "Mais típicos de quem recomenda", CORES["Recomenda"], True),
]):
    valores = dados["razao"] if not crescente else 1 / dados["razao"]
    ax.barh(dados["termo"], valores, color=cor)
    ax.invert_yaxis()
    ax.set_title(titulo)
    ax.set_xlabel("Quantas vezes o termo é mais frequente na classe")

plt.tight_layout()
salvar(fig, "03_vocabulario_classes")
plt.show()

## 5. Categorias de produto

In [ ]:
categorias = (df.groupby("site_category_lv1")
              .agg(avaliacoes=("recomenda", "size"), taxa_recomenda=("recomenda", "mean"))
              .query("avaliacoes >= 200")
              .sort_values("taxa_recomenda"))

fig, ax = plt.subplots(figsize=(10, max(4, 0.32 * len(categorias))))
cores = plt.cm.RdYlGn(categorias["taxa_recomenda"] / categorias["taxa_recomenda"].max())
ax.barh(categorias.index, categorias["taxa_recomenda"], color=cores)
ax.axvline(df["recomenda"].mean(), color="black", ls="--", lw=1,
           label=f"média geral: {df['recomenda'].mean():.0%}")
ax.set_title("Taxa de recomendação por categoria")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend()
salvar(fig, "04_categorias")
plt.show()

categorias.assign(taxa_recomenda=lambda d: (d["taxa_recomenda"] * 100).round(1)).head(10)

## 6. Qualidade dos dados

Três checagens que costumam revelar problemas em bases de avaliações: textos duplicados
(mesmo comentário copiado em vários produtos), avaliações muito curtas e casos contraditórios
(nota alta com "não recomendo", ou o contrário).

In [ ]:
duplicados = df["review_text"].duplicated().sum()
curtas = (df["n_palavras"] <= 2).sum()
contraditorias = ((df["overall_rating"] >= 4) & (df["recomenda"] == 0)).sum() + \
                 ((df["overall_rating"] <= 2) & (df["recomenda"] == 1)).sum()

pd.Series({
    "textos repetidos": duplicados,
    "avaliações com até 2 palavras": curtas,
    "nota e recomendação contraditórias": contraditorias,
    "total de avaliações": len(df),
})

In [ ]:
print("Exemplos de contradição (nota alta, mas não recomenda):")
exemplos = df[(df["overall_rating"] >= 4) & (df["recomenda"] == 0)]
exemplos[["overall_rating", "review_title", "review_text"]].head(5)

Esses casos não são erro de digitação: em geral o cliente gostou do produto e reclamou da entrega,
ou o contrário. São exemplos legitimamente difíceis, e é bom saber disso antes de olhar as métricas —
nenhum modelo vai acertar 100% num conjunto que contém contradições reais.

Textos repetidos, por outro lado, precisam de cuidado: se o mesmo comentário aparecer no treino e no teste,
a avaliação do modelo fica otimista demais. Vamos remover as duplicatas exatas.

## 7. Base preparada

In [ ]:
base = (df.drop_duplicates(subset=["review_text"])
        .loc[df["n_palavras"] >= 3,
             ["submission_date", "texto", "texto_limpo", "n_palavras", "recomenda",
              "overall_rating", "site_category_lv1", "reviewer_state"]]
        .reset_index(drop=True))

print(f"Base final: {len(base):,} avaliações")
print(f"Taxa de recomendação: {base['recomenda'].mean():.1%}")

base.to_csv(PASTA_PROCESSED / "avaliacoes_preparadas.csv", index=False, encoding="utf-8")
print("Salvo em", PASTA_PROCESSED / "avaliacoes_preparadas.csv")

## Conclusões

- O alvo escolhido é **recomendaria a um amigo**, binário e com leitura de negócio direta.
- As classes são desbalanceadas, então acurácia sozinha não serve como métrica: vamos usar
  precisão, recall e F1 da classe minoritária, além da curva ROC.
- Quem não recomenda escreve textos mais longos, e o vocabulário das duas classes é bem distinto —
  bom sinal para um modelo linear sobre TF-IDF.
- Existem contradições legítimas entre nota e recomendação, que estabelecem um teto natural de acerto.

**Próximo passo:** `02_classificacao.ipynb` — TF-IDF com regressão logística como baseline,
comparação com outros modelos, análise de erros e interpretação dos pesos.